# 04 - SwinUNETR Explained

**Goal:** Understand SwinUNETR - your production model architecture.

---

## What is SwinUNETR?

**Swin UNET R**esidual = Swin Transformer encoder + U-Net decoder

```
SwinUNETR = Swin Transformer (encoder) + CNN (decoder) + Skip connections
```

Combines:
- **Swin Transformer**: Global context, efficient attention
- **U-Net**: Proven decoder for segmentation, skip connections
- **Residual connections**: Better gradient flow

## Architecture Diagram

```
Input Volume (96×96×96×1)
        │
        ▼
┌───────────────────────────────────────────────────────────────┐
│                    SWIN TRANSFORMER ENCODER                    │
├───────────────────────────────────────────────────────────────┤
│  Patch Partition (2×2×2)                                      │
│        │                                                       │
│        ▼                                                       │
│  Stage 1: Swin Blocks ──────────────────────┐  Skip 1 (48³×24)│
│        │                                     │                 │
│  Patch Merge                                 │                 │
│        │                                     │                 │
│  Stage 2: Swin Blocks ──────────────┐       │  Skip 2 (24³×48)│
│        │                             │       │                 │
│  Patch Merge                         │       │                 │
│        │                             │       │                 │
│  Stage 3: Swin Blocks ──────┐       │       │  Skip 3 (12³×96)│
│        │                     │       │       │                 │
│  Patch Merge                 │       │       │                 │
│        │                     │       │       │                 │
│  Stage 4: Swin Blocks        │       │       │  Skip 4 (6³×192)│
│        │                     │       │       │                 │
└────────┼─────────────────────┼───────┼───────┼─────────────────┘
         │ Bottleneck          │       │       │
         ▼                     │       │       │
┌───────────────────────────────────────────────────────────────┐
│                       CNN DECODER                              │
├───────────────────────────────────────────────────────────────┤
│  Upsample + Conv ◄───────────┘       │       │                 │
│        │                             │       │                 │
│  Upsample + Conv ◄───────────────────┘       │                 │
│        │                                     │                 │
│  Upsample + Conv ◄───────────────────────────┘                 │
│        │                                                       │
│  Upsample + Conv ◄───────────────────────────────────────────  │
│        │                                                       │
└────────┼───────────────────────────────────────────────────────┘
         │
         ▼
   Final Conv (1×1×1)
         │
         ▼
Output (96×96×96×num_classes)
```

In [ ]:
import torch
from monai.networks.nets import SwinUNETR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Create SwinUNETR exactly like your production code

model = SwinUNETR(
    img_size=(96, 96, 96),    # Input volume size
    in_channels=1,            # Grayscale CT
    out_channels=4,           # Number of classes
    feature_size=24,          # Base feature channels (C in diagram)
    use_checkpoint=False,     # Gradient checkpointing (saves memory)
)

print(f"Model created!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test forward pass
x = torch.randn(1, 1, 96, 96, 96)  # Batch=1, Channel=1, 96³ volume

model.eval()
with torch.no_grad():
    output = model(x)

print(f"Input: {x.shape}")
print(f"Output: {output.shape}")
print(f"\nOutput is logits for {output.shape[1]} classes at each voxel")

## Understanding the Components

Let's look inside the model:

In [ ]:
# Print model structure
print("=== SwinUNETR Components ===")
print()
for name, module in model.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"{name}: {params:,} parameters")

In [ ]:
# Look at the Swin encoder
print("=== Swin Encoder (swinViT) ===")
print()
for name, module in model.swinViT.named_children():
    params = sum(p.numel() for p in module.parameters())
    if params > 0:
        print(f"{name}: {params:,} parameters")

## Feature Sizes Through the Network

For 96×96×96 input with feature_size=24:

In [ ]:
# Trace feature sizes
print("Feature sizes through SwinUNETR (96³ input, feature_size=24):")
print()
print("ENCODER (Swin Transformer):")
print(f"  Input:       96×96×96 ×  1 channel")
print(f"  Stage 1:     48×48×48 × 24 channels  (skip connection 1)")
print(f"  Stage 2:     24×24×24 × 48 channels  (skip connection 2)")
print(f"  Stage 3:     12×12×12 × 96 channels  (skip connection 3)")
print(f"  Stage 4:      6× 6× 6 ×192 channels  (skip connection 4)")
print(f"  Bottleneck:   3× 3× 3 ×384 channels")
print()
print("DECODER (CNN):")
print(f"  Up + Skip 4:  6× 6× 6 ×192 channels")
print(f"  Up + Skip 3: 12×12×12 × 96 channels")
print(f"  Up + Skip 2: 24×24×24 × 48 channels")
print(f"  Up + Skip 1: 48×48×48 × 24 channels")
print(f"  Final Up:    96×96×96 × num_classes")

## Your Production Configuration

From `segmentation/config/spine_sequential_1mm_azure_v6.yaml`:

```yaml
model:
  input_shape: [128, 128, 96]  # Input patch size
  features_chan: 24            # Swin feature size
  
testing:
  inference:
    simultaneous_patch: 4      # Process 4 patches at once
    patch_overlap: 0.5         # 50% overlap between patches
```

The sliding window inference handles volumes larger than 128×128×96 by processing overlapping patches.

In [ ]:
# Simulate your production inference
from monai.inferers import sliding_window_inference

# Create a model matching your production config
model = SwinUNETR(
    img_size=(128, 128, 96),  # Your instance model size
    in_channels=1,
    out_channels=4,
    feature_size=24,
)
model.eval()

# Simulate a larger input volume
large_volume = torch.randn(1, 1, 256, 256, 192)

print(f"Input volume: {large_volume.shape}")
print(f"Model patch size: (128, 128, 96)")
print()

# This is exactly what your production code does
with torch.no_grad():
    output = sliding_window_inference(
        inputs=large_volume,
        roi_size=(128, 128, 96),  # Patch size
        sw_batch_size=4,          # Process 4 patches at once
        predictor=model,
        overlap=0.5,              # 50% overlap
    )

print(f"Output: {output.shape}")
print("\nSliding window handles arbitrary input sizes!")

## Why SwinUNETR for Medical Imaging?

| Feature | Benefit for Spine Segmentation |
|---------|-------------------------------|
| **Swin encoder** | Global context - sees entire spine structure |
| **Hierarchical features** | Captures both large vertebrae and fine boundaries |
| **U-Net decoder** | Precise localization with skip connections |
| **3D native** | Works directly on CT volumes |
| **Efficient attention** | Handles large 3D volumes |
| **Sliding window** | Processes any size input |

## Connection to Your Production Code

```python
# From inference_engine_v016.py

from monai.networks.nets import SwinUNETR
from monai.inferers import sliding_window_inference

# Create model
model = SwinUNETR(
    img_size=config["model"]["input_shape"],    # [128, 128, 96]
    in_channels=1,
    out_channels=nb_classes,
    feature_size=config["model"]["features_chan"],  # 24
)

# Load trained weights
model.load_state_dict(torch.load(weights_path))

# Inference
with torch.amp.autocast('cuda'):  # Mixed precision for speed
    pred = sliding_window_inference(
        data["image"].unsqueeze(0),
        roi_size=config["model"]["input_shape"],
        sw_batch_size=config["testing"]["inference"]["simultaneous_patch"],
        predictor=model,
        overlap=config["testing"]["inference"]["patch_overlap"],
    )
```

**Now you understand every line of this!**

## Summary

| Component | What you learned |
|-----------|------------------|
| **Phase 0** | Neural networks, training, CNNs |
| **Phase 1** | PyTorch fundamentals |
| **Phase 2** | Segmentation, U-Net, MONAI |
| **Phase 3** | Attention, ViT, Swin, SwinUNETR |

You now understand your production spine segmentation pipeline from fundamentals to architecture!

---

## Phase 3 Complete!

**Next: Phase 4** - MLOps (DVC, MLflow) to version data and track experiments.